# E7 step 3c: C4's counting on the fair questions

**What:** goal 2 promised building counting above the base 7B's 34.2% on the benchmark's counting questions (BDC). C4's map scores 35.34% (E7 step 3), so goal 2 is met as written. But on these questions a rule that never looks at the photo scores 66.98%: when a question's largest option is above 20, the middle option is always the right one. So this notebook scores every counter on the **fair** questions only, those whose largest option is 20 or less, where that pattern doesn't hold. L2 step 4b used the same split for the 3B on 23 Nov.

**Compared:** C4 (blobs, and its building table), R11 alone, find-then-grade alone (seed 42), the blind rule (largest option above 20: the middle value; otherwise the second smallest), our 7B (L0 step 2) and the 3B (c) at normal precision and in 4 bits. Every answer comes from saved files: `E7-results/step3b/counting_per_question.csv` (the counters, 3,580 questions with readable photos) and `L2-step4-results/full/checks/counting_per_question.csv` (the 7B and the 3B).

**The rule, stated 25 Nov before computing (the same test as L2 step 4b):** C4's lead over 34.2% is more than the options' pattern only if, on the fair questions, C4 beats the blind rule by more than chance (exact McNemar, p < 0.05).

**Honest note:** the numbers were first seen in a quick check in the session the same day; this notebook reproduces and keeps them. No expectation was fixed in advance, so none is claimed.

**Also reported:** the best rule that picks by option position alone (smallest, second smallest and so on), since a smarter blind rule could explain a lead; and how close each counter's count is to the true count (C4's raw count is credited for being close even when the nearest option is wrong).

**Not possible here:** the base 7B on the fair questions. Its 34.2% is the paper's figure; its answers per question aren't saved.

In [1]:
# ── Cell 1: The saved answers, joined by question ──
import ast, math, re
from pathlib import Path
import numpy as np, pandas as pd

CT = Path.cwd()
OUT = CT / "E7-results" / "step3c"
OUT.mkdir(parents=True, exist_ok=True)
e7 = pd.read_csv(CT / "E7-results" / "step3b" / "counting_per_question.csv").set_index("id")
ck = pd.read_csv(CT / "L2-step4-results" / "full" / "checks" / "counting_per_question.csv").set_index("id")
d = e7.join(ck[ck["task"] == "BDC"][["values", "part", "answer_c", "answer_4bit", "answer_7b", "correct_7b", "correct_3b", "correct_4bit"]], how="inner")
d["fair"] = d["largest_option"] <= 20
assert len(d) == 3580, len(d)
assert (d["fair"] == (d["part"] == "fair")).all(), "the two files disagree on which questions are fair"
d["vals"] = d["values"].map(ast.literal_eval)
COUNTERS = {"C4 (blobs)": "C4 blobs | correct", "C4 (building table)": "C4 table | correct", "R11 alone": "R11 | correct",
            "find-then-grade alone": "chain_s42 | correct", "blind rule": "no_image_rule | correct", "our 7B": "correct_7b",
            "3B (c)": "correct_3b", "3B (c), 4 bits": "correct_4bit"}
for col in COUNTERS.values():
    d[col] = d[col].astype(bool)
print(f"{len(d):,} building-counting questions; {int(d['fair'].sum()):,} fair (largest option 20 or less)")


3,580 building-counting questions; 2,000 fair (largest option 20 or less)


In [2]:
# ── Cell 2: Accuracy (%) on all, fair, fair and clean, and pattern questions ──
SUBSETS = {"all": d, "fair": d[d["fair"]], "fair, clean": d[d["fair"] & d["e5_clean"]], "pattern (largest above 20)": d[~d["fair"]]}
acc = pd.DataFrame({name: {k: round(100 * s[c].mean(), 2) for k, c in COUNTERS.items()} | {"questions": len(s)}
                    for name, s in SUBSETS.items()}).T
print(acc.to_string())


                            C4 (blobs)  C4 (building table)  R11 alone  find-then-grade alone  blind rule  our 7B  3B (c)  3B (c), 4 bits  questions
all                              35.34                33.66      33.02                  32.88       66.98   30.34   49.83           48.21     3580.0
fair                             42.20                42.10      41.55                  40.25       40.90   32.00   57.45           53.90     2000.0
fair, clean                      41.59                41.49      40.93                  40.05       40.87   31.52   57.22           53.62     1945.0
pattern (largest above 20)       26.65                22.97      22.22                  23.54      100.00   28.23   40.19           41.01     1580.0


In [3]:
# ── Cell 3: Exact McNemar tests on the fair questions ──
def mcnemar(a, b):
    x, y = int((a & ~b).sum()), int((~a & b).sum())
    n = x + y
    return x, y, (1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, k) for k in range(min(x, y) + 1)) / 2 ** n))

f = SUBSETS["fair"]
PAIRS = [("C4 (blobs)", "blind rule"), ("C4 (blobs)", "our 7B"), ("3B (c)", "C4 (blobs)"), ("3B (c)", "blind rule"),
         ("R11 alone", "blind rule"), ("find-then-grade alone", "blind rule"), ("C4 (building table)", "blind rule")]
rows = []
for a, b in PAIRS:
    x, y, p = mcnemar(f[COUNTERS[a]], f[COUNTERS[b]])
    rows.append({"A": a, "B": b, "A_%": round(100 * f[COUNTERS[a]].mean(), 2), "B_%": round(100 * f[COUNTERS[b]].mean(), 2),
                 "only_A_right": x, "only_B_right": y, "p": float(f"{p:.3g}")})
tests = pd.DataFrame(rows)
P_RULE = tests.loc[0, "p"]
PASSES = bool(tests.loc[0, "A_%"] > tests.loc[0, "B_%"] and P_RULE < 0.05)
print(tests.to_string(index=False))
print(f"\nThe rule: C4 beats the blind rule on the fair questions by more than chance? {'passes' if PASSES else 'fails'} (p = {P_RULE})")


                    A          B   A_%  B_%  only_A_right  only_B_right            p
           C4 (blobs) blind rule 42.20 40.9           585           559 4.600000e-01
           C4 (blobs)     our 7B 42.20 32.0           519           315 1.620000e-12
               3B (c) C4 (blobs) 57.45 42.2           616           311 6.990000e-24
               3B (c) blind rule 57.45 40.9           616           285 9.790000e-29
            R11 alone blind rule 41.55 40.9           573           560 7.210000e-01
find-then-grade alone blind rule 40.25 40.9           574           587 7.250000e-01
  C4 (building table) blind rule 42.10 40.9           585           561 4.970000e-01

The rule: C4 beats the blind rule on the fair questions by more than chance? fails (p = 0.46)


In [4]:
# ── Cell 4: Rules that pick by option position alone, on the fair questions ──
ranks = pd.Series([sorted(v).index(t) + 1 for v, t in zip(f["vals"], f["truth"])], index=f.index)
position = pd.DataFrame({"option by size (1 = smallest)": range(1, 6),
                         "right_%": [round(100 * (ranks == r).mean(), 1) for r in range(1, 6)]})
BEST_POSITION = position["right_%"].max()
print(position.to_string(index=False), f"\nbest position-only rule: {BEST_POSITION}%")


 option by size (1 = smallest)  right_%
                             1     26.6
                             2     40.9
                             3     17.6
                             4     10.4
                             5      4.4 
best position-only rule: 40.9%


In [5]:
# ── Cell 5: How close each count is to the true count, on the fair questions ──
def letters(text):
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

def chosen(answer, vals):
    g = letters(answer)
    return vals["ABCDE".index(next(iter(g)))] if len(g) == 1 and next(iter(g)) in "ABCDE"[:len(vals)] else np.nan

def blind(vals):
    s = sorted(vals)
    return s[2] if s[4] > 20 else s[1]

counts = {"C4 raw count": f["C4 blobs | count"],
          "C4 nearest option": pd.Series([min(v, key=lambda x: abs(x - c)) for v, c in zip(f["vals"], f["C4 blobs | count"])], index=f.index),
          "blind rule": pd.Series([blind(v) for v in f["vals"]], index=f.index),
          "our 7B": pd.Series([chosen(a, v) for a, v in zip(f["answer_7b"], f["vals"])], index=f.index),
          "3B (c)": pd.Series([chosen(a, v) for a, v in zip(f["answer_c"], f["vals"])], index=f.index)}
error = pd.DataFrame([{"count from": k, "median_error": (c - f["truth"]).abs().median(), "mean_error": round((c - f["truth"]).abs().mean(), 2),
                       "exact_%": round(100 * ((c - f["truth"]).abs() == 0).mean(), 1), "answered_%": round(100 * c.notna().mean(), 1)}
                      for k, c in counts.items()])
print(f"true counts on the fair questions: median {f['truth'].median():.0f}, mean {f['truth'].mean():.1f}\n")
print(error.to_string(index=False))


true counts on the fair questions: median 4, mean 5.8

       count from  median_error  mean_error  exact_%  answered_%
     C4 raw count           2.0        3.78     13.2       100.0
C4 nearest option           1.0        2.46     42.2       100.0
       blind rule           1.0        2.20     40.9       100.0
           our 7B           3.0        4.55     32.0       100.0
           3B (c)           0.0        1.95     57.4       100.0


In [6]:
# ── Cell 6: The files and the README ──
def md(df, index=True):
    t = df.reset_index() if index else df
    head = "| " + " | ".join(map(str, t.columns)) + " |\n|" + "---|" * len(t.columns)
    return head + "".join("\n| " + " | ".join(map(str, r)) + " |" for r in t.itertuples(index=False))

acc.to_csv(OUT / "accuracy_by_subset.csv")
tests.to_csv(OUT / "fair_tests.csv", index=False)
position.to_csv(OUT / "position_rules.csv", index=False)
error.to_csv(OUT / "count_error.csv", index=False)
c4, rule, b3 = acc.loc["fair", "C4 (blobs)"], acc.loc["fair", "blind rule"], acc.loc["fair", "3B (c)"]
readme = f'''# E7 step 3c: C4's counting on the fair questions

From `e7-step3c-fair-counting.ipynb`, run on the PC from saved answers (no GPU). Fair: the building-counting questions whose largest option is 20 or less ({len(f):,} of {len(d):,}), where the middle option isn't always right. The rule and an honest note on timing are in the notebook's first cell.

## Accuracy (%)

{md(acc)}

## The rule: does C4 beat the blind rule on the fair questions?

{md(tests, index=False)}

**{"Passes" if PASSES else "Fails"}:** on the fair questions C4 scores {c4}% against the blind rule's {rule}% (p = {P_RULE}). {"Its counting lead is more than the options' pattern." if PASSES else "So its 35.34% on all questions, above the promised 34.2%, can't show that it counts: on the fair questions it is no better than a rule that never sees the photo. It is ahead of our 7B."}

The 3B (c) scores {b3}% on the fair questions, ahead of C4 and of the blind rule beyond chance.

## A smarter blind rule?

{md(position, index=False)}

No rule that picks by option position alone gets above {BEST_POSITION}% on the fair questions, so the 3B's lead is more than a position pattern.

## How close the counts are (fair questions; true counts median {f["truth"].median():.0f}, mean {f["truth"].mean():.1f})

{md(error, index=False)}

## Files

`accuracy_by_subset.csv`, `fair_tests.csv`, `position_rules.csv`, `count_error.csv`
'''
(OUT / "README.md").write_text(readme, encoding="utf-8")
print(readme)


# E7 step 3c: C4's counting on the fair questions

From `e7-step3c-fair-counting.ipynb`, run on the PC from saved answers (no GPU). Fair: the building-counting questions whose largest option is 20 or less (2,000 of 3,580), where the middle option isn't always right. The rule and an honest note on timing are in the notebook's first cell.

## Accuracy (%)

| index | C4 (blobs) | C4 (building table) | R11 alone | find-then-grade alone | blind rule | our 7B | 3B (c) | 3B (c), 4 bits | questions |
|---|---|---|---|---|---|---|---|---|---|
| all | 35.34 | 33.66 | 33.02 | 32.88 | 66.98 | 30.34 | 49.83 | 48.21 | 3580.0 |
| fair | 42.2 | 42.1 | 41.55 | 40.25 | 40.9 | 32.0 | 57.45 | 53.9 | 2000.0 |
| fair, clean | 41.59 | 41.49 | 40.93 | 40.05 | 40.87 | 31.52 | 57.22 | 53.62 | 1945.0 |
| pattern (largest above 20) | 26.65 | 22.97 | 22.22 | 23.54 | 100.0 | 28.23 | 40.19 | 41.01 | 1580.0 |

## The rule: does C4 beat the blind rule on the fair questions?

| A | B | A_% | B_% | only_A_right | only_B